# Class-wise normalization for standard (chunked) inference — fairness baseline (PLAN Amendment I)

**How to run:** Runtime → Change runtime type → **L4 GPU** (T4 also works, slower) → Save. Then Runtime → **Run all**. Allow Google Drive.

* About 1.5–2 hours on an L4 (~3 hours on a T4). Keep the tab open.
* Saved to `MyDrive/ovd/calib_chunks_v1/` as it goes. **If Colab disconnects, press Run all again** — it continues.
* At the end it prints `ALL DONE` and downloads one or more files named `calib_chunks_v1_partXofN.bin` (each under 25 MB). **Upload all of them.**

What it runs: the standard full-vocabulary Grounding DINO inference (1,203 classes in 14 fixed chunks) on 300 label-free reference images (to measure each class's score scale) and on the 1,000 full-image test images, saving the top-300 lists with and without the same class-wise normalization that is being tested for the bank. This tells whether the normalization also helps the standard inference (a fair comparison).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, os, subprocess
print('GPU:', torch.cuda.get_device_name() if torch.cuda.is_available() else 'NONE - change runtime to a GPU')
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> L4 GPU'
os.makedirs('/content/drive/MyDrive/ovd', exist_ok=True)
subprocess.run('rm -rf /content/Research && git clone -q -b claude/beautiful-brown-gdjzcq https://github.com/apersonli55555/Research.git /content/Research', shell=True, check=True)
print(subprocess.run('cd /content/Research && git log -1 --format=%h', shell=True, capture_output=True, text=True).stdout)

In [ ]:
!cd /content/Research && PYTHONUNBUFFERED=1 python ovd/bank/bank_calib.py --method chunks --mode ref --out /content/drive/MyDrive/ovd/calib_chunks_v1 --images /content/drive/MyDrive/ovd/images 2>&1 | grep --line-buffered -v "Loading weights"
!cd /content/Research && PYTHONUNBUFFERED=1 python ovd/bank/bank_calib.py --method chunks --mode test --out /content/drive/MyDrive/ovd/calib_chunks_v1 --images /content/drive/MyDrive/ovd/images 2>&1 | grep --line-buffered -v "Loading weights"


In [ ]:
import glob, math, os, zipfile
from google.colab import files
root = '/content/drive/MyDrive/ovd/calib_chunks_v1'
assert len(glob.glob(root + '/chunks/test/*.json')) == 1000, 'test not finished - run the cell above again'
zp = '/content/calib_chunks_v1.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for p in sorted(glob.glob(root + '/**/*', recursive=True)):
        if os.path.isfile(p) and not p.endswith('.tmp'):
            z.write(p, os.path.relpath(p, os.path.dirname(root)))
data = open(zp, 'rb').read(); n = max(1, math.ceil(len(data) / 25e6)); size = -(-len(data) // n)
for i in range(n):
    p = f'/content/calib_chunks_v1_part{i + 1}of{n}.bin'; open(p, 'wb').write(data[i * size:(i + 1) * size]); files.download(p)
print('ALL DONE', len(data) / 1e6, 'MB in', n, 'part(s)')
